# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Identifier: {metadata.identifier}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview
Review available record sets and their fields. Each is uniquely referenced by its `@id` field.

In [ ]:
# List the record sets with their @id and fields
def print_record_sets_and_fields(dataset):
    print("Record Sets (@id):\n" + "="*40)
    for record_set in dataset.record_sets:
        print(f"@id: {record_set['@id']}")
        print(f"  Name: {record_set.get('name', '')}")
        print(f"  Description: {record_set.get('description', '')}")
        print("  Fields:")
        for field in record_set['fields']:
            print(f"    - @id: {field['@id']} | name: {field.get('name', '')} | dataType: {field.get('dataType', '')}")
        print("")

print_record_sets_and_fields(dataset)

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Reference all entities by their `@id` values as listed above.

In [ ]:
# Set the @id values for the available record sets you want to load
# You can use the list output from above to choose record sets for analysis
# Example assignment (change these to match your dataset's actual @id values):
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            # Print the columns for this record set
            print(f"Columns for record set {record_set_id}:")
            print(dataframes[record_set_id].columns.tolist())
            print(dataframes[record_set_id].head())
    except Exception as e:
        print(f"Error loading record set {record_set_id}: {e}")

# For the rest of this notebook, we'll use the first record set for illustration
main_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None

## 4. Exploratory Data Analysis (EDA)
Let's perform basic data filtering and normalization on a numeric field. We'll reference all field/@id and column names as printed in the previously loaded DataFrame.

In [ ]:
# Choose a numeric field (e.g., 'MW_kDa') and a grouping field (e.g., 'description') by their @id or name
df = dataframes[main_record_set_id] if main_record_set_id in dataframes else pd.DataFrame()

if not df.empty:
    # Guessing likely fields - please adjust as needed for your dataset
    # Try to find numeric column candidates
    numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    if not numeric_candidates:
        # Try to convert possible numeric columns
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except:
                continue
        numeric_candidates = df.select_dtypes(include=['number']).columns.tolist()
    print("Numeric candidates:", numeric_candidates)
    # Select the first numeric field for demonstration
    numeric_field = numeric_candidates[0] if numeric_candidates else None
    if numeric_field:
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a text field
        group_candidates = df.select_dtypes(include=['object']).columns.tolist()
        group_field = group_candidates[0] if group_candidates else None
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}:\n", grouped_df.head())
else:
    print("No records loaded to analyze.")

## 5. Visualization
Visualize the distribution and relationships of numeric fields using simple plots. Adjust fields as desired.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not df.empty and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. We reviewed the metadata, inspected available record sets and their fields using their `@id`s, extracted tabular data, performed basic cleaning and normalization, and visualized key variables. 

To further analyze, reference additional record sets or fields using their `@id` as shown above, and tailor the analysis to your study needs.